# 🥫 Aula 19 — Proporção, variância e tamanho de amostra numa fábrica de latas do Sul de Minas

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **controle de qualidade de uma fábrica de latas de alumínio em Pouso Alegre**, no Sul de Minas. A fábrica quer responder três perguntas, na ordem da Aula 19:

1. que **proporção** das latas sai com defeito? (IC para a proporção $p$)
2. o intervalo para a proporção **cumpre os 95%**? (otimista × conservador e a condição $np \ge 5$)
3. quanto a **espessura** das latas varia? (IC para a variância $\sigma^2$ e a importância da normalidade)
4. **quantas latas** inspecionar para ter a margem de erro desejada? (tamanho de amostra)

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **O que o código faz**, antes de cada célula;
4. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude). Confira antes de usar.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores, e os sorteios só saem iguais aos do texto se a ordem for respeitada.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.07` = 0,07). Números como `2.874e-05` querem dizer $2{,}874 \times 10^{-5}$.

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats`**: as **tabelas** da normal (`norm`) e da qui-quadrado (`chi2`) dentro do computador. `norm.ppf(0.975)` dá $z_{0{,}025} = 1{,}96$, e `chi2.isf(0.025, 19)` dá $\chi^2_{0{,}025;19} = 32{,}852$ (área 0,025 à **direita**, como na tabela de cauda superior da Aula 17).

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Só o notebook sabe** os valores verdadeiros: **8%** das latas saem com defeito ($p = 0{,}08$), e a espessura da parede tem média 0,100 mm e desvio padrão $\sigma = 0{,}005$ mm. A fábrica não conhece esses valores — é isso que ela quer estimar.

In [ ]:
import numpy as np                     # calculadora de números e sorteios
import matplotlib.pyplot as plt        # gráficos
from scipy.stats import norm, chi2     # as "tabelas"
rng = np.random.default_rng(42)        # sorteador, semente 42
p_real = 0.08                          # proporção verdadeira de latas com defeito (a fábrica NÃO conhece)
mu_esp, sigma_esp = 0.100, 0.005       # espessura verdadeira: média e desvio padrão (mm)
print("Tudo pronto!")

## 1. Que proporção das latas sai com defeito?

### A situação
O controle de qualidade sorteia **$n = 300$** latas da produção do dia e conta quantas têm defeito.

### A pergunta
Qual é o **IC de 95%** para a proporção $p$ de latas com defeito?

### A ferramenta da aula
$$IC(p;\,1-\alpha) = \left[\hat p - z_{\alpha/2}\sqrt{\frac{\hat p(1-\hat p)}{n}}\ ;\ \hat p + z_{\alpha/2}\sqrt{\frac{\hat p(1-\hat p)}{n}}\right]$$
Vale se $n\hat p \ge 5$ e $n(1-\hat p) \ge 5$. O **conservador** troca $\hat p(1-\hat p)$ por $1/4$: $\hat p \pm z_{\alpha/2}\,\dfrac{1}{2\sqrt n}$.

### O que o código faz
1. sorteia as 300 latas e calcula $\hat p$;
2. confere a condição;
3. calcula o erro padrão, a margem de erro e os dois intervalos (otimista e conservador).

In [ ]:
n = 300
latas = rng.random(n) < p_real
p_chapeu = latas.mean()
print(f"Latas com defeito na amostra: {latas.sum()} de {n}")
print(f"Proporção amostral: {p_chapeu:.4f}")
print(f"Condição: n*p_chapeu = {n * p_chapeu:.0f} e n*(1 - p_chapeu) = {n * (1 - p_chapeu):.0f}")
ep = np.sqrt(p_chapeu * (1 - p_chapeu) / n)
margem = norm.ppf(0.975) * ep
print(f"Erro padrão: {ep:.4f}   Margem de erro: {margem:.4f}")
print(f"IC de 95% (otimista):    [{p_chapeu - margem:.4f} ; {p_chapeu + margem:.4f}]")
margem_c = norm.ppf(0.975) / (2 * np.sqrt(n))
print(f"IC de 95% (conservador): [{p_chapeu - margem_c:.4f} ; {p_chapeu + margem_c:.4f}]")

**Como ler:**
- A amostra teve **21 latas com defeito**: $\hat p = 0{,}07$. A condição vale (21 e 279 são maiores que 5).
- O IC otimista é cerca de **[0,041 ; 0,099]**: entre 4% e 10% das latas. Ele **contém** o valor verdadeiro $p = 0{,}08$, mas a fábrica não sabe disso.
- O conservador, **[0,013 ; 0,127]**, é bem mais largo: ele usa $p(1-p) = 0{,}25$, mas aqui $\hat p(1-\hat p) \approx 0{,}065$. Quando $p$ está longe de 0,5, o conservador fica **desnecessariamente largo**.

## 2. O intervalo para a proporção cumpre os 95%?

### A situação
Imagine repetir a inspeção **100.000 vezes**, cada vez com uma amostra nova, e construir os dois intervalos em cada uma.

### A pergunta
Que porcentagem dos intervalos contém o valor verdadeiro $p = 0{,}08$? E como isso muda com o tamanho da amostra?

### A ferramenta da aula
"95% de confiança" quer dizer que cerca de **95% dos intervalos** construídos assim contêm $p$ (Aula 18). O intervalo otimista depende da **aproximação normal**, que só é boa se $np \ge 5$ e $n(1-p) \ge 5$.

### O que o código faz
Para $n$ = 20, 60, 300 e 1000 latas: sorteia 100.000 amostras, constrói os dois intervalos e conta quantos contêm $p$.

In [ ]:
def cobertura(n, n_sim=100_000):
    pc = rng.binomial(n, p_real, n_sim) / n
    m_otim = 1.96 * np.sqrt(pc * (1 - pc) / n)
    m_cons = 1.96 / (2 * np.sqrt(n))
    return (np.abs(pc - p_real) <= m_otim).mean(), (np.abs(pc - p_real) <= m_cons).mean()
print("    n   n*p   otimista   conservador")
for n_ in (20, 60, 300, 1000):
    o, c = cobertura(n_)
    print(f"{n_:5d}  {n_ * p_real:4.1f}    {o:6.1%}      {c:6.1%}")

**Como ler:**
- Com **$n = 20$** ($np = 1{,}6$) e **$n = 60$** ($np = 4{,}8$), a condição **falha**, e o intervalo otimista acerta só **81% e 86%** das vezes: promete 95% e entrega bem menos.
- Com $n = 300$ ($np = 24$), ele chega a **93,6%**, e com $n = 1000$, a **95,1%**. Quando $p$ está perto de 0 (aqui, 8%), a aproximação normal precisa de amostras maiores para funcionar bem; a condição $np \ge 5$ é o **mínimo**.
- O **conservador** fica sempre acima de 95% (quase 100%): ele garante a confiança, mas, com $p$ longe de 0,5, à custa de um intervalo muito largo.

## 3. Quanto a espessura das latas varia?

### A situação
A espessura da parede tem distribuição **normal**. O cliente exige latas parecidas entre si, então a fábrica quer estimar a **variabilidade** $\sigma^2$. Ela mede **$n = 20$** latas.

### A pergunta
Qual é o **IC de 95%** para $\sigma^2$ e para $\sigma$?

### A ferramenta da aula
Numa amostra de população normal, $(n-1)S^2/\sigma^2$ tem distribuição $\chi^2(n-1)$ (Aula 17). Daí:
$$IC(\sigma^2;\,1-\alpha) = \left[\frac{(n-1)s^2}{\chi^2_{\alpha/2;\,n-1}}\ ;\ \frac{(n-1)s^2}{\chi^2_{1-\alpha/2;\,n-1}}\right]$$
O IC para $\sigma$ é a raiz dos dois limites. Com $n = 20$: $\chi^2_{0{,}025;19} = 32{,}852$ e $\chi^2_{0{,}975;19} = 8{,}907$ (os mesmos do Exemplo 2 da aula, a laminação de Timóteo).

### O que o código faz
1. sorteia as 20 espessuras e calcula $s^2$;
2. lê os dois valores da tabela qui-quadrado;
3. calcula os intervalos para $\sigma^2$ e para $\sigma$.

In [ ]:
n = 20
esp = rng.normal(mu_esp, sigma_esp, n)
s2 = esp.var(ddof=1)
lim_inf_tab, lim_sup_tab = chi2.isf(0.025, n - 1), chi2.isf(0.975, n - 1)
print(f"Variância amostral: s² = {s2:.3e} mm²  (s = {np.sqrt(s2):.5f} mm)")
print(f"Tabela: chi2(0,025; {n-1}) = {lim_inf_tab:.3f}   chi2(0,975; {n-1}) = {lim_sup_tab:.3f}")
L, U = (n - 1) * s2 / lim_inf_tab, (n - 1) * s2 / lim_sup_tab
print(f"IC de 95% para sigma²: [{L:.3e} ; {U:.3e}] mm²")
print(f"IC de 95% para sigma:  [{np.sqrt(L):.5f} ; {np.sqrt(U):.5f}] mm")

**Como ler:**
- A amostra deu $s \approx 0{,}00536$ mm, e o IC de 95% para $\sigma$ é cerca de **[0,00408 ; 0,00783] mm**. Ele contém o valor verdadeiro $\sigma = 0{,}005$.
- Repare que o intervalo **não é simétrico** em torno de $s$: de 0,00408 até 0,00536 são 0,00128 mm; de 0,00536 até 0,00783, 0,00247 mm. É a assimetria da qui-quadrado.
- O **maior** valor da tabela (32,852) dá o limite **inferior**.

## 3b. E se a espessura não for normal?

### A pergunta
O IC para $\sigma^2$ supõe população **normal**. Se a espessura tiver uma distribuição **assimétrica** (com a mesma média e o mesmo desvio padrão), o intervalo ainda acerta 95% das vezes? E com amostras maiores?

### O que o código faz
Para $n$ = 20, 50 e 200: sorteia 100.000 amostras de uma população normal e 100.000 de uma população assimétrica (uma exponencial deslocada, como as da Aula 12), constrói o IC de 95% para $\sigma^2$ em cada uma e conta quantos contêm o valor verdadeiro $\sigma^2$.

In [ ]:
def cobertura_var(gerador, n, n_sim=100_000):
    x = gerador((n_sim, n))
    s2 = x.var(axis=1, ddof=1)
    L = (n - 1) * s2 / chi2.isf(0.025, n - 1)
    U = (n - 1) * s2 / chi2.isf(0.975, n - 1)
    return ((L <= sigma_esp**2) & (sigma_esp**2 <= U)).mean()
normal = lambda tam: rng.normal(mu_esp, sigma_esp, tam)
assimetrica = lambda tam: mu_esp - sigma_esp + rng.exponential(sigma_esp, tam)
print("    n    normal   assimétrica")
for n_ in (20, 50, 200):
    print(f"{n_:5d}   {cobertura_var(normal, n_):6.1%}     {cobertura_var(assimetrica, n_):6.1%}")

**Como ler:**
- Com população **normal**, o intervalo cumpre os **95%** em todos os tamanhos de amostra.
- Com população **assimétrica**, acerta só cerca de **73%** com $n = 20$, e **não melhora** com $n$ maior (70% com $n = 50$, 68% com $n = 200$).
- É o que diz a aula: o IC para a variância é **pouco robusto** à falta de normalidade, e aumentar $n$ não resolve (diferente do IC da média, em que o TLC ajuda). Antes de usar esse intervalo, olhe o **histograma** dos dados.

## 4. Quantas latas inspecionar?

### A situação
Antes da próxima inspeção, a fábrica quer escolher o tamanho da amostra para ter uma margem de erro desejada $E$, com 95% de confiança.

### A ferramenta da aula
- Proporção: $n = \left(\dfrac{z_{\alpha/2}}{E}\right)^2 p(1-p)$, com $p$ de uma pesquisa anterior; sem informação, $p = 0{,}5$ (o maior $n$).
- Média: $n = \left(\dfrac{z_{\alpha/2}\,\sigma}{E}\right)^2$.
- Arredonde **sempre para cima**.

### O que o código faz
Monta duas tabelas: o $n$ para a proporção de latas com defeito (usando $p = 0{,}08$ e usando $p = 0{,}5$) e o $n$ para a espessura média, para várias margens de erro.

In [ ]:
z = norm.ppf(0.975)
print("Proporção de latas com defeito (95% de confiança)")
print("  margem E   n (p = 0,08)   n (sem informação, p = 0,5)")
for E in (0.05, 0.03, 0.02, 0.01):
    n_p = np.ceil((z / E)**2 * 0.08 * 0.92)
    n_c = np.ceil((z / E)**2 * 0.25)
    print(f"   {E:.2f}        {n_p:6.0f}           {n_c:6.0f}")
print()
print("Espessura média (sigma = 0,005 mm, 95% de confiança)")
for E in (0.002, 0.001, 0.0005):
    print(f"   margem {E} mm  ->  n = {np.ceil((z * sigma_esp / E)**2):.0f} latas")

**Como ler:**
- Com a informação de que $p \approx 0{,}08$, bastam **315 latas** para margem 0,03; sem informação ($p = 0{,}5$), seriam **1.068**. Conhecer $p$ economiza muitas inspeções.
- Margem pela **metade** (de 0,02 para 0,01): o $n$ fica **4 vezes** maior (707 → 2.828). O mesmo vale para a média (97 → 385 latas quando a margem cai de 0,001 para 0,0005 mm).

## 4b. Com esse $n$, a margem fica mesmo em 0,02?

### A pergunta
A fábrica escolheu **$n = 707$** latas, calculado com $p = 0{,}08$ para margem 0,02. Como $\hat p$ muda de amostra para amostra, a margem $1{,}96\sqrt{\hat p(1-\hat p)/n}$ também muda. Ela fica sempre abaixo de 0,02?

### O que o código faz
Sorteia 100.000 amostras de 707 latas, calcula a margem de cada uma e conta em quantas ela ficou em até 0,02.

In [ ]:
n = int(np.ceil((z / 0.02)**2 * 0.08 * 0.92))
pc = rng.binomial(n, p_real, 100_000) / n
margens = z * np.sqrt(pc * (1 - pc) / n)
print(f"n = {n}: margem média {margens.mean():.4f}; margem até 0,02 em {(margens <= 0.02).mean():.0%} das amostras")
print(f"Intervalos que contêm p: {(np.abs(pc - p_real) <= margens).mean():.1%}")

**Como ler:**
- A margem fica, **em média**, em 0,02 (0,0199), mas passa de 0,02 em cerca de **metade** das amostras: quando $\hat p$ sai maior que 0,08, a margem sai um pouco maior.
- Isso não é um erro: com $p = 0{,}08$, o $n$ foi calculado para que a margem fique **em torno** de 0,02. Se a fábrica quiser **garantir** a margem, usa um $p$ maior (ou $p = 0{,}5$, a abordagem conservadora), ao custo de inspecionar mais latas.
- Os intervalos continuam contendo $p$ em cerca de 95% das amostras.

## Para fechar

Neste notebook, o controle de qualidade da fábrica de latas usou as ferramentas da Aula 19:

- **Proporção:** $\hat p \pm z_{\alpha/2}\sqrt{\hat p(1-\hat p)/n}$, que só cumpre os 95% quando a aproximação normal é boa ($np \ge 5$ e $n(1-p) \ge 5$ é o mínimo); o conservador garante a confiança, mas pode ficar muito largo;
- **Variância:** o intervalo com a qui-quadrado é assimétrico e só vale para população **normal** — aumentar $n$ não resolve;
- **Tamanho de amostra:** $n = (z_{\alpha/2}/E)^2 p(1-p)$ ou $n = (z_{\alpha/2}\sigma/E)^2$, sempre arredondado para cima; margem pela metade, $n$ 4 vezes maior.

Na **Aula 20**, as mesmas distribuições servem para outra pergunta: decidir entre duas hipóteses sobre um parâmetro (teste de hipóteses).